<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-H3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎬 MiniMax-H3 — Video + Audio Generation (33B, INT8 quantized)

A Colab port of [MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) — a 33B parameter video generation model that produces **video with synchronized audio** (ambience, foley, speech). Supports text-to-video, image-to-video (first/last frame), and reference-based generation.

## How it works

MiniMax-H3 uses a **split deployment** architecture:

1. **Conditioner** — encodes the text prompt + optional keyframe images into `prompt_embeds` + `text_token_tags`. Two modes:
   - **Remote** (default): calls the `multimodalart/qwen3vl-conditioner` HF Space via `gradio_client`. No local text encoder needed (saves 62 GB download + VRAM). Depends on HuggingFace ZeroGPU quota.
   - **Local**: loads the 62 GB Qwen3-VL text encoder locally with INT8 quantization (~31 GB), runs the conditioning step, then frees the text encoder before loading the transformer. No ZeroGPU dependency. Requires downloading an additional 62 GB of weights (cached on Drive).

2. **Local denoiser** (on your Colab GPU): the 33B transformer (DiT) denoises the video+audio latents, then the VAEs decode them into frames + stereo audio. Uses the official diffusers `minimax-h3` branch with **INT8 quantization + block-level group offloading** (from the [official docs](https://github.com/huggingface/diffusers/blob/minimax-h3/docs/source/en/api/pipelines/minimax_h3.md)).

```
prompt + images → [remote conditioner HF Space] → prompt_embeds
                                                    ↓
              INT8 DiT (group_offload) → denoise → VAEs → video.mp4 + audio
```

## ⚠️ License — Territory Restriction + MAU Cap

MiniMax H3 Community License:
- **Excludes:** EU, UK, South Korea, **and USA**
- **>1M MAU** requires separate commercial license
- Continuing past the header cell is your acceptance of the license

## Quick start

1. **Runtime → Change runtime type → GPU** (L4, A100, or A100 80GB)
2. Run **STEP 1** — installs torch 2.11.0+cu128, diffusers from `minimax-h3` branch, torchao for INT8 quantization. First run: ~10-15 min.
3. Run **STEP 2** — downloads FL2VA transformer (61.7 GB) + VAEs (10.3 GB) to Drive cache. First run: ~30-60 min.
4. Run **STEP 3** — imports, remote conditioner, lazy model loader with INT8 + group offload
5. Run **STEP 4** — opens the Gradio UI. Enter a prompt, pick canvas/duration/steps, click Generate.
6. **STEP 5** keep-alive, **STEP 6** quick test, **STEP 7** batch

## Memory

The official docs provide recipes for different GPU sizes:

| GPU | VRAM | Recipe | Peak VRAM |
|-----|------|--------|-----------|
| **A100 80GB** | 80 GB | `auto_cpu_offload` (no quantization) | ~68 GB |
| **A100 40GB** | 40 GB | INT8 + `group_offload(block_level)` | ~18 GB |
| **L4 22GB** | 22 GB | INT8 + `group_offload(block_level)` + small canvas | ~15 GB |

The notebook auto-detects GPU VRAM and picks the right recipe. Smaller canvases (960×544) run ~2.3x faster per step than the trained 1344×768.

## Outputs

```
output.mp4    # Video + synchronized stereo audio (32 kHz)
```

## Technical notes

- **diffusers from `minimax-h3` branch**: MiniMax-H3 support is in an open PR (#14355), not on PyPI. We install from the branch directly.
- **INT8 quantization**: Uses `TorchAoConfig(Int8WeightOnlyConfig(version=2))` from the official docs. The `version=2` tensors are pinnable, which streamed offload needs.
- **Block-level group offloading**: `enable_group_offload(offload_type="block_level", num_blocks_per_group=1)` moves one transformer block at a time to GPU. This is the key to fitting on L4/A100-40GB.
- **Remote conditioner**: The 62 GB Qwen3-VL text encoder runs on the HF Space. We call it via `gradio_client`. This saves 62 GB download + 62 GB VRAM.
- **`spaces` stub**: The upstream code uses `import spaces` for HF ZeroGPU. We install a stub module so the code runs on Colab.
- **PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True**: Reduces memory fragmentation.

## Companion notebooks

- **Wan2.2_Colab** — text/image-to-video (no audio)
- **Wan2.2_Animate_Colab** — character animation
- **GaussianGPT_Colab** — autoregressive 3D scene generation
- **InfiniSplat_Colab** — single-image 3DGS reconstruction


In [ ]:
#@title STEP 1 — Install torch 2.11.0+cu128, diffusers PR #14355 head, torchao, transformers
"""
• Pins torch to 2.11.0+cu128 (the MiniMax-H3 release needs a recent torch)
• Installs diffusers from PR #14355 head (MiniMax-H3 is now in MODULAR_PIPELINE_MAPPING,
  so the package-level `MiniMaxH3ModularPipeline` export resolves correctly)
• Installs torchao for INT8 quantization (Int8WeightOnlyConfig)
• Pins transformers 5.8.0 (required for Qwen3-VL processor)
• Installs PyAV for video+audio muxing
• Stubs the `spaces` module (HF ZeroGPU API, not available on Colab)
• Mounts Google Drive for checkpoint caching
    • Picks a loading recipe (auto / bf16-a100 / int8-prequant / int8-prequant-split /
    int8-remote-text). The recipe has to match your GPU + host RAM;
    auto reads both and picks the safest.
"""
import os, sys, time, subprocess, pathlib, types

print('='*72)
print('MiniMax-H3 — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/MiniMax-H3')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Drive cache  : {drive_root}')
else:
    drive_root = pathlib.Path('/content/_h3_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}')

OUT_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/MiniMax-H3')
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT = pathlib.Path('/content/h3_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Loading recipe — pick one that fits your hardware.
#   auto            : detect GPU VRAM + host RAM and pick a recipe automatically.
#   bf16-a100       : A100 80GB (or any 80GB card). bfloat16 + auto_cpu_offload.
#                     Requires ≥75 GB host RAM for the bf16 staging during load.
#   sdnq-4bit       : 24 GB VRAM (L4) + ≥80 GB host RAM. Loads the prequantized
#                     `OzzyGT/MiniMax_H3_sdnq_dynamic_4bit` checkpoint (37 GB on
#                     disk: 17.5 GB transformer + 19.6 GB text encoder). Both
#                     components fit resident on ≥80 GB hosts; on 53 GB Colab
#                     Pro+ the text encoder's bf16 keep-in-fp32 modules push the
#                     resident total over 53 GB and the recipe OOMs at the
#                     denoise step.
#   sdnq-8bit       : ≥40 GB VRAM + ≥100 GB host RAM. Same checkpoint family as
#                     `sdnq-4bit` but 8-bit dynamic quant (94 GB on disk, better
#                     quality than 4-bit).
#   int8-prequant-split : 24-40 GB VRAM + ≥45 GB host RAM. INT8 conditioner
#                     + INT8 denoiser pipelines from the upstream split pattern
#                     (`h3_split_blocks.py`); they are run sequentially so both
#                     fit on L4 + 53 GB host RAM without OOM. **Default on L4.**
#   int8-remote-text : 24 GB VRAM (L4) + 50 GB host RAM. INT8 denoiser
#                     pipeline only; skips loading the 62 GB Qwen3-VL text encoder
#                     and uses the upstream `multimodalart/qwen3vl-conditioner` HF
#                     Space for prompt encoding (ZeroGPU quota, ~5 min/day on free).
LOAD_MODE = 'auto'  #@param ['auto', 'bf16-a100', 'sdnq-4bit', 'sdnq-8bit', 'int8-prequant-split', 'int8-remote-text']
os.environ['H3_LOAD_MODE'] = LOAD_MODE
print(f'  Load mode     : {LOAD_MODE}')

t_total = time.time()

# 1. Pin torch to 2.11.0+cu128 ───────────────────────────────────────────
TARGET_TORCH = '2.11.0'
if not torch.__version__.startswith(TARGET_TORCH):
    print(f'\n[1/5] Pinning torch to {TARGET_TORCH}+cu128 ...')
    t0 = time.time()
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--disable-pip-version-check', '--no-input',
        f'torch=={TARGET_TORCH}+cu128',
        'torchvision==0.26.0+cu128',
        'torchaudio==2.11.0+cu128',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
        '--force-reinstall',
    ], check=False)
    print(f'  torch installed in {time.time()-t0:.1f}s')
    print('  Restarting kernel to load new torch ...')
    import os as _os
    _os.execv(sys.executable, [sys.executable] + sys.argv)
else:
    print(f'\n[1/5] torch {torch.__version__} already matches — fixing huggingface_hub')
    t0 = time.time()
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'huggingface-hub>=1.25.0',
    ], check=False)
    print(f'  huggingface-hub>=1.25.0 installed in {time.time()-t0:.1f}s')

# 2. Install diffusers from main (gets MiniMax-H3 + PR #14398 SDNQ activation-dtype fix). ──
# The upstream docs note (PR #14401) recommends installing from main until MiniMax-H3
# is part of a diffusers release. PR #14398 (merged 2026-08-05) fixes a casting bug
# where SDNQ-quantized weights' activation dtype mismatches the projection's compute
# dtype — required for loading `OzzyGT/MiniMax_H3_sdnq_dynamic_4bit` cleanly.
print('\n[2/5] Installing diffusers from main (MiniMax-H3 + SDNQ fix) ...')
t0 = time.time()
# Use --no-deps to prevent diffusers from pulling conflicting deps.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'git+https://github.com/huggingface/diffusers.git',
], check=False)
print(f'  diffusers installed in {time.time()-t0:.1f}s')

# 3. Install torchao + transformers + accelerate + other deps ───────────
# Force-reinstall torchao to rebuild against the new torch (after os.execv restart).
print('\n[3/5] Installing torchao + transformers + accelerate ...')
t0 = time.time()
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torchao>=0.15.0', '--force-reinstall', '--no-deps',
], check=False)
EXTRA_PKGS = [
    'transformers==5.8.0',
    'accelerate==1.14.0',
    'huggingface-hub>=1.25.0',
    'safetensors>=0.8.0',
    'av',
    'einops',
    'omegaconf',
    'gradio>=5.49.1,<7',
    'pillow',
    'scipy',
    'opencv-python-headless',
    'psutil',
    'sdnq>=0.2.3',
    'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_PKGS, check=False)
print(f'  Extra deps installed in {time.time()-t0:.1f}s')

# 4. Stub the `spaces` module (HF ZeroGPU, not available on Colab) ─────
print('\n[4/5] Stubbing `spaces` module ...')
spaces_stub = types.ModuleType('spaces')
def _gpu_decorator(duration=120, size=None):
    def decorator(fn):
        return fn
    return decorator
spaces_stub.GPU = _gpu_decorator
sys.modules['spaces'] = spaces_stub
print('  spaces stub installed')

# 5. Verify imports ─────────────────────────────────────────────────────
print('\n[5/5] Verifying imports ...')
t0 = time.time()
try:
    import diffusers
    print(f'  diffusers    : {diffusers.__version__}')
except ImportError as e:
    print(f'  [FAIL] diffusers: {e}')
try:
    import torchao
    print(f'  torchao      : {torchao.__version__}')
except ImportError as e:
    print(f'  [WARN] torchao: {e}')
try:
    import transformers
    print(f'  transformers  : {transformers.__version__}')
except ImportError as e:
    print(f'  [FAIL] transformers: {e}')
try:
    import av
    print(f'  av           : OK')
except ImportError as e:
    print(f'  [FAIL] av: {e}')

elapsed = time.time() - t_total
print()
print('='*72)
print(f'STEP 1 complete in {elapsed/60:.1f} min')
print('='*72)
print(f'  Drive cache  : {drive_root}')
print(f'  Output dir   : {OUT_DIR}')
print()
print('Next: run STEP 2 (download FL2VA weights).')


In [ ]:
#@title STEP 2 — Download weights to Drive cache (bf16 OR SDNQ)
"""
Downloads the MiniMax-H3 weights to Drive cache. The download set depends on
LOAD_MODE (set in STEP 1):

  - `bf16-a100` / `int8-prequant-split` / `int8-remote-text`: the full bf16
    checkpoint from `MiniMaxAI/MiniMax-H3` (~134 GB on disk — both transformer
    partitions, both VAEs, the Qwen3-VL text encoder, schedulers, tokenizer).
  - `sdnq-4bit` / `sdnq-8bit`: the prequantized SDNQ checkpoint from
    `OzzyGT/MiniMax_H3_sdnq_dynamic_{4bit,8bit}` (37 GB / 94 GB on disk) — the
    weights are already quantized, so no on-the-fly quant is needed at load time.

When LOAD_MODE='auto' the recipe is resolved by STEP 3's `_resolve_load_mode()`
and the corresponding download set is used here. Note that `auto` may pick a
different recipe than the one used at download time if the host RAM changes
between sessions — re-run STEP 2 if the picked recipe and the download set disagree.
"""
import os, sys, time, pathlib
from huggingface_hub import snapshot_download

print('='*72)
print('MiniMax-H3 — Download weights')
print('='*72)

CKPT_DIR = drive_root / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve the LOAD_MODE once here so we pick the right repo.
# STEP 3 redoes this resolution, but we need it now to choose the download set.
_LOAD_MODE = os.environ.get('H3_LOAD_MODE', 'auto')
import psutil as _ps
_HOST_RAM_GB = int(_ps.virtual_memory().total // (1024**3))

# Same auto-priority order as STEP 3's `_resolve_load_mode`
# bf16-a100, sdnq-8bit, sdnq-4bit, int8-prequant-split, int8-remote-text
def _pick_download_repo():
    if torch.cuda.is_available():
        _vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    else:
        _vram = 0
    if _LOAD_MODE == 'auto':
        if _vram >= 70 and _HOST_RAM_GB >= 75:
            return ('MiniMaxAI/MiniMax-H3', 'bf16-a100')
        if _HOST_RAM_GB >= 95 and _vram >= 24:
            return ('OzzyGT/MiniMax_H3_sdnq_dynamic_8bit', 'sdnq-8bit')
        if _HOST_RAM_GB >= 80 and _vram >= 20:
            return ('OzzyGT/MiniMax_H3_sdnq_dynamic_4bit', 'sdnq-4bit')
        if _vram >= 20 and _HOST_RAM_GB >= 45:
            # L4 + Colab Pro+ (53 GB host RAM): sdnq-4bit OOMs here (text
            # encoder's bf16 keep-in-fp32 modules push the resident total over
            # 53 GB). Use the int8-prequant-split recipe instead, which
            # INT8-quantizes the bf16 checkpoint on the way in and runs the
            # conditioner + denoiser sequentially so peak host RAM is ~40 GB.
            return ('MiniMaxAI/MiniMax-H3', 'int8-prequant-split')
        if _vram >= 20:
            return ('MiniMaxAI/MiniMax-H3', 'int8-remote-text')
        raise RuntimeError(
            f'No viable recipe for GPU={_vram:.0f}GB host RAM={_HOST_RAM_GB}GB. '
            'MiniMax-H3 needs ≥20 GB VRAM.'
        )
    return {
        'bf16-a100':             ('MiniMaxAI/MiniMax-H3', 'bf16-a100'),
        'sdnq-4bit':             ('OzzyGT/MiniMax_H3_sdnq_dynamic_4bit', 'sdnq-4bit'),
        'sdnq-8bit':             ('OzzyGT/MiniMax_H3_sdnq_dynamic_8bit', 'sdnq-8bit'),
        'int8-prequant-split':   ('MiniMaxAI/MiniMax-H3', 'int8-prequant-split'),
        'int8-remote-text':      ('MiniMaxAI/MiniMax-H3', 'int8-remote-text'),
    }[_LOAD_MODE]

MODEL_REPO, _ = _pick_download_repo()
# Persist the resolved repo id so STEP 3's loader points at the same checkpoint.
os.environ['H3_MODEL_REPO'] = MODEL_REPO
SDNQ_CHECKPOINT = MODEL_REPO != 'MiniMaxAI/MiniMax-H3'

print(f'  Load mode     : {_LOAD_MODE}')
print(f'  Model repo    : {MODEL_REPO}')
print(f'  Cache dir     : {CKPT_DIR}')
print()

if SDNQ_CHECKPOINT:
    # SDNQ checkpoint — only the FL2VA workflow files (no transformer_ref, no .bin). Smaller than bf16.
    ALLOW_PATTERNS = [
        'modular_model_index.json',
        # FL2VA transformer (sharded, ~17.5 GB for 4-bit, ~37 GB for 8-bit)
        'transformer/*.safetensors',
        'transformer/config.json',
        'transformer/*.index.json',
        # Video VAE (~9.7 GB bf16)
        'vae/*.safetensors',
        'vae/config.json',
        'vae/*.index.json',
        # Audio VAE (~577 MB)
        'audio_vae/*.safetensors',
        'audio_vae/config.json',
        # Text encoder (Qwen3-VL, already SDNQ-quantized on disk)
        'text_encoder/*.safetensors',
        'text_encoder/config.json',
        'text_encoder/generation_config.json',
        # Tokenizer / processor (tiny JSON)
        'tokenizer/*.json',
        'tokenizer/*.txt',
        'processor/*.json',
        'processor/*.txt',
        # Schedulers (tiny JSON files)
        'scheduler/scheduler_config.json',
        'audio_scheduler/scheduler_config.json',
    ]
    print('  Downloading SDNQ prequantized checkpoint (no transformer_ref) ...')
else:
    # Full bf16 checkpoint set: both transformer partitions + both VAEs + text_encoder.
    ALLOW_PATTERNS = [
        'modular_model_index.json',
        # FL2VA transformer (14 shards, ~61.7 GB)
        'transformer/*.safetensors',
        'transformer/config.json',
        'transformer/*.index.json',
        # Ref2VA transformer (separate partition, ~61.7 GB)
        'transformer_ref/*.safetensors',
        'transformer_ref/config.json',
        'transformer_ref/*.index.json',
        # Video VAE (~9.7 GB)
        'vae/*.safetensors',
        'vae/config.json',
        'vae/*.index.json',
        # Audio VAE (~577 MB)
        'audio_vae/*.safetensors',
        'audio_vae/config.json',
        # Text encoder (~62 GB bf16 — streamed off CPU on smaller cards)
        'text_encoder/*.safetensors',
        'text_encoder/config.json',
        'text_encoder/*.index.json',
        'text_encoder/chat_template.json',
        'text_encoder/preprocessor_config.json',
        'text_encoder/video_preprocessor_config.json',
        # Tokenizer / processor (tiny JSON)
        'tokenizer/*.json',
        'tokenizer/*.txt',
        'processor/*.json',
        'processor/*.txt',
        # Schedulers (tiny JSON files)
        'scheduler/scheduler_config.json',
        'audio_scheduler/scheduler_config.json',
    ]
    print('  Downloading full bf16 checkpoint (both transformer partitions) ...')

t_total = time.time()
print(f'  Downloading {len(ALLOW_PATTERNS)} pattern groups ...')
if SDNQ_CHECKPOINT:
    if '4bit' in MODEL_REPO:
        print('  (This downloads ~37 GB. First run: 15-25 min depending on network.)')
    else:
        print('  (This downloads ~94 GB. First run: 30-50 min depending on network.)')
else:
    print('  (This downloads ~134 GB. First run: 30-60 min depending on network.)')
print()

snapshot_download(
    repo_id=MODEL_REPO,
    allow_patterns=ALLOW_PATTERNS,
    local_dir=str(CKPT_DIR),
    cache_dir=os.environ.get('HF_HOME'),
    max_workers=4,
)

elapsed = time.time() - t_total

# Verify downloads
print()
print('  Downloaded files:')
total_size = 0
for f in sorted(CKPT_DIR.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size
        total_size += sz
        rel = str(f.relative_to(CKPT_DIR))
        if sz > 1024**3:
            print(f'    {rel:>55s}  {sz/1024**3:.2f} GB')
        elif sz > 1024**2:
            print(f'    {rel:>55s}  {sz/1024**2:.1f} MB')

print()
print('='*72)
print(f'STEP 2 complete in {elapsed/60:.1f} min')
print(f'  Total downloaded: {total_size/1024**3:.1f} GB')
print(f'  Cache dir: {CKPT_DIR}')
print('='*72)
print()
print('Next: run STEP 3 (imports + lazy model loader).')


In [ ]:
#@title STEP 3 — Imports, spaces stub, lazy model loader + generate_video
"""
• Stubs the `spaces` module (HF ZeroGPU, not available on Colab)
• Defines `_load_sdnq_pipe`, `_load_conditioner_pipe`, `_load_denoiser_pipe`, `_load_bf16_pipe` —
  per-recipe loaders; the right one is selected by LOAD_MODE.
• Defines `generate_video()` — runs the chosen recipe end-to-end:
  condition → denoise → decode → mux video+audio.
  Recipes are:
    - 'bf16-a100'           : single bf16 + auto_cpu_offload on A100 80GB.
    - 'sdnq-4bit'           : loads the prequantized
      `OzzyGT/MiniMax_H3_sdnq_dynamic_4bit` checkpoint (37 GB on disk).
      Recommended on L4 + 53 GB host RAM — both components fit resident.
    - 'sdnq-8bit'           : 8-bit SDNQ checkpoint (94 GB on disk).
    - 'int8-prequant-split' : INT8 conditioner + INT8 denoiser pipelines, run
      sequentially so both fit on L4 + 53 GB host RAM without OOM.
    - 'int8-remote-text'    : INT8 denoiser pipeline + upstream HF Space
      `multimodalart/qwen3vl-conditioner` for prompt encoding (ZeroGPU quota).
• Auto-detects GPU VRAM + host RAM and picks a recipe when LOAD_MODE='auto'.
"""
import os, sys, time, gc, pathlib, types, traceback
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
import psutil

print('='*72)
print('MiniMax-H3 — Imports + lazy model loader')
print('='*72)

# --- Stub the 'spaces' module (must come before any diffusers import) ──
_spaces_stub = types.ModuleType('spaces')
def _gpu_decorator(duration=120, size=None):
    def decorator(fn):
        return fn
    return decorator
_spaces_stub.GPU = _gpu_decorator
sys.modules['spaces'] = _spaces_stub

# --- Verify deps ────────────────────────────────────────────────────────
import diffusers
import transformers
try:
    import torchao
    print(f'  diffusers    : {diffusers.__version__}')
    print(f'  transformers  : {transformers.__version__}')
    print(f'  torchao      : {torchao.__version__}')
except ImportError as e:
    print(f'  [FAIL] {e}')
    raise
print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU_VRAM_GB = p.total_memory / (1024**3)
    print(f'  GPU          : {p.name}  ({GPU_VRAM_GB:.1f} GB)')
else:
    GPU_VRAM_GB = 0
    print('  WARNING: no GPU detected')
print()

CKPT_DIR = drive_root / 'checkpoints'
# MODEL_REPO comes from H3_MODEL_REPO env var (set by STEP 2 once it has resolved
# the SDNQ vs bf16 checkpoint). Fall back to the bf16 release for safety.
MODEL_REPO = os.environ.get('H3_MODEL_REPO', 'MiniMaxAI/MiniMax-H3')
CONDITIONER_SPACE = 'multimodalart/qwen3vl-conditioner'

    # --- Canvas definitions (resolution + aspect-ratio presets) ──────────
CANVASES = {
    "960x544 · 16:9 fast": (544, 960),
    "1024x576 · 16:9 fast": (576, 1024),
    "1152x640 · 16:9": (640, 1152),
    "1280x704 · 16:9": (704, 1280),
    "1344x768 · 16:9 full": (768, 1344),
    "544x960 · 9:16 fast": (960, 544),
    "640x1152 · 9:16": (1152, 640),
    "768x1344 · 9:16 full": (1344, 768),
    "544x544 · 1:1 fast": (544, 544),
    "768x768 · 1:1 full": (768, 768),
    "768x576 · 4:3 fast": (576, 768),
    "1024x768 · 4:3 full": (768, 1024),
    "576x768 · 3:4 fast": (768, 576),
    "768x1024 · 3:4 full": (1024, 768),
    "1152x512 · 21:9 fast": (512, 1152),
    "1536x672 · 21:9 full": (672, 1536),
}
DEFAULT_CANVAS = "960x544 · 16:9 fast"
FPS, FRAMES_PER_CHUNK, LATENTS_PER_CHUNK = 24, 17, 5
MIN_UI_DURATION = 2  # Lowered from 5s to 2s so the L4 + int8-prequant-split
                     # recipe can fit shorter clips. `MiniMaxH3ModularPipeline.min_duration`
                     # is also patched to 2.0 inside the loaders so the pipeline
                     # block accepts these requests.
MAX_UI_DURATION = 14

def snap_frames(seconds):
    """The frame count MiniMax-H3's video VAE can decode: the next 17*n + 5 at 24 fps."""
    frames = max(1, round(float(seconds) * FPS))
    while frames % FRAMES_PER_CHUNK != LATENTS_PER_CHUNK:
        frames += 1
    return frames

# --- Conditioner dispatch (placeholder; the pipeline's text_encoder step does it now) ──
def call_conditioner(*args, **kwargs):
    """No-op stub: the official MiniMax-H3 pipeline runs Qwen3-VL internally.

    Kept so older UI code paths don't break. Returns the prompt unchanged.
    """
    return None, None, {'height': '0', 'width': '0', 'num_frames': '0', 'prompt': ''}, {}

# --- Load mode resolution --------------------------------------------------
# LOAD_MODE is set in STEP 1 and persisted via H3_LOAD_MODE env var (survives the os.execv kernel restart).
# It is resolved against the host's hardware here to the concrete recipe we'll use.
LOAD_MODE = os.environ.get('H3_LOAD_MODE', 'auto')
_HOST_RAM_GB = int(psutil.virtual_memory().total // (1024**3))

def _resolve_load_mode():
    """Map a LOAD_MODE setting + observed hardware to a concrete recipe.

    Recipes (host RAM budget includes the framework, model state, and safetensors mmap):
      'bf16-a100'           : ComponentsManager + auto_cpu_offload on the official single pipeline.
                              Needs ≥75 GB host RAM for bf16 staging (134 GB on disk).
      'sdnq-4bit'           : Loads the prequantized `OzzyGT/MiniMax_H3_sdnq_dynamic_4bit`
                              checkpoint (37 GB on disk). Text encoder on host is dominated
                              by the keep-in-fp32 modules (~14 GB bf16 vision tower +
                              embed_tokens + lm_head), plus ~5 GB int4 LLM, plus 17.5 GB
                              int4 transformer, plus ~10.5 GB bf16 VAEs. Needs ≥80 GB host
                              RAM (host RAM OOMs at 53 GB Colab Pro+).
      'sdnq-8bit'           : Same as `sdnq-4bit` but the 8-bit dynamic variant (94 GB on
                              disk) — for ≥100 GB host RAM hosts.
      'int8-prequant-split' : Two ModularPipelines — conditioner (text_encoder + tiny components)
                              and denoiser (transformer + tiny components) — built from the official
                              blocks. Run conditioner, save state to Drive, free, run denoiser.
                              Each is INT8-quantized on the way in; both fit in 53 GB host RAM
                              because they never coexist. **Default on L4 + Colab Pro+.**
      'int8-remote-text'    : Single denoiser pipeline; prompt encoding via gradio_client to
                              the upstream `multimodalart/qwen3vl-conditioner` HF Space.
    """
    if LOAD_MODE == 'auto':
        if GPU_VRAM_GB >= 70 and _HOST_RAM_GB >= 75:
            return 'bf16-a100'
        if _HOST_RAM_GB >= 95 and GPU_VRAM_GB >= 24:
            # A100 node with 256 GB host RAM, or workstation with 96+ GB.
            return 'sdnq-8bit'
        if _HOST_RAM_GB >= 80 and GPU_VRAM_GB >= 20:
            # L4 with Pro+++ (or similar): full sdnq-4bit fits.
            # The text encoder is ~19.6 GB on host (LLM 4-bit + vision tower bf16),
            # the transformer is ~17.5 GB, VAEs ~10.5 GB, ~47 GB resident + overhead.
            return 'sdnq-4bit'
        if GPU_VRAM_GB >= 20 and _HOST_RAM_GB >= 45:
            # L4 + Colab Pro+ (53 GB host RAM): the SDNQ-4bit text encoder's
            # keep-in-fp32 modules (visual tower, embed_tokens, lm_head) total
            # ~14 GB of bf16 plus ~5 GB of int4 LLM weights, and combined with
            # the 17.5 GB int4 transformer + 10.5 GB bf16 VAEs we need ~50 GB
            # resident on host. With the ComponentsManager auto-cpu-offload the
            # peak VRAM is fine, but the resident on host OOMs at 53 GB.
            #
            # Use the sequential INT8 quant recipe instead: transformer bf16
            # quantizes to ~30 GB int8 on host, text encoder bf16 quantizes to
            # ~30 GB int8 on host. They are never resident together (we load,
            # run text encoder, save state, free, load denoiser, run), so peak
            # host RAM is ~40 GB (30 GB int8 + 10 GB bf16 VAEs).
            return 'int8-prequant-split'
        if GPU_VRAM_GB >= 20:
            return 'int8-remote-text'
        raise RuntimeError(
            f'No viable recipe for GPU={GPU_VRAM_GB:.0f}GB host RAM={_HOST_RAM_GB}GB. '
            'MiniMax-H3 needs ≥20 GB VRAM.'
        )
    if LOAD_MODE in ('bf16-a100', 'sdnq-4bit', 'sdnq-8bit', 'int8-prequant-split', 'int8-remote-text'):
        return LOAD_MODE
    raise ValueError(f'Unknown LOAD_MODE: {LOAD_MODE!r}')

RECIPE = _resolve_load_mode()
print(f'  Recipe resolved: {RECIPE}  (GPU {GPU_VRAM_GB:.0f}GB, host RAM {_HOST_RAM_GB}GB)')

# Serialise shard loads. Diffusers' threadpool reads up to 8 shards (~4 GB each) into
# host RAM concurrently — 32 GB of read buffers — which OOMs at 53 GB host RAM even with
# 'device_map='cpu''. 'num_workers=1' keeps peak RAM at one shard + accumulating INT8 model.
import diffusers.utils.constants as _dutils_constants
_dutils_constants.DEFAULT_HF_PARALLEL_LOADING_WORKERS = 1

# Bind the streaming-load to CPU with a budget so the bf16 stage never spills onto GPU.
# Without this, transformers' low_cpu_mem_usage=True (auto-set on quant) materializes the
# full bf16 weight on the device — 61.7 GB for the transformer alone, OOM-ing on any GPU.
_CPU_MEM_BUDGET = f'{max(8, _HOST_RAM_GB - 8)}GiB'
MAX_MEMORY = {'cpu': _CPU_MEM_BUDGET}

# Modules to keep in bf16 (not quantized). Per the diffusers MiniMax-H3 docs recipe.
_TRANSFORMER_KEEP_BF16 = [
    'proj_in', 'audio_proj_in', 'context_embedder', 'time_embedder', 'time_proj',
    'token_refiner', 'norm_out', 'proj_out', 'audio_proj_out',
]
_TEXT_ENCODER_KEEP_BF16 = [
    'model.visual', 'model.language_model.embed_tokens',
    'model.language_model.norm', 'lm_head',
]

# --- Vendor 'h3_split_blocks.py' (the split-pattern blocks) -------------------
# The split pattern — 'MiniMaxH3ConditionerBlocks' (Qwen3-VL + keyframes) and
# 'MiniMaxH3GeneratorBlocks' (denoiser + decode) — is what the upstream ZeroGPU Space
# uses, and it's what lets us run text_encoder and denoiser on the same L4 + 53 GB host
# by loading each component only when it's needed. We write a CORRECTED copy of that
# file to WORK_ROOT at runtime and import it from there.
#
# Why vendored, not downloaded from the upstream Space: the upstream Space's
# h3_split_blocks.py imports `MiniMaxH3AutoKeyframeVaeEncoderStep` and
# `MiniMaxH3AutoResizeStep` from `modular_blocks_minimax_h3`, but those names are from
# an earlier revision. In the current `main` (and on the diffusers revision we pin),
# the equivalent classes are `MiniMaxH3AutoVaeEncoderStep` (wraps the keyframe +
# ref2va branches) and `MiniMaxH3AutoBeforeEncodeStep` (wraps `MiniMaxH3Ref2VASetupStep`
# and `MiniMaxH3ResizeStep`). So fetching the upstream file ImportErrors on the first
# line. We reproduce the upstream split pattern here, with the renames applied.
_SPLIT_BLOCKS_LOCAL = WORK_ROOT / 'h3_split_blocks.py'
_SPLIT_BLOCKS_SOURCE = '''"""The halves of a split MiniMax-H3 deployment, for both of its checkpoint partitions.

Corrected for diffusers `main` (post 2026-08-05): upstream classes MiniMaxH3AutoResizeStep
and MiniMaxH3AutoKeyframeVaeEncoderStep renamed to MiniMaxH3AutoBeforeEncodeStep and
MiniMaxH3AutoVaeEncoderStep respectively. Local helper _generation_outputs() replaces the
private upstream one with a public equivalent.
"""
from diffusers.modular_pipelines.minimax_h3.before_encoder import MiniMaxH3Ref2VASetupStep
from diffusers.modular_pipelines.minimax_h3.decoders import MiniMaxH3AfterDenoiseStep
from diffusers.modular_pipelines.minimax_h3.encoders import (
    MiniMaxH3Ref2VAReferenceEncoderStep,
    MiniMaxH3Ref2VATextEncoderStep,
    MiniMaxH3TextEncoderStep,
)
from diffusers.modular_pipelines.minimax_h3.modular_blocks_minimax_h3 import (
    MiniMaxH3AutoBeforeEncodeStep,
    MiniMaxH3AutoVaeEncoderStep,
    MiniMaxH3CoreDenoiseStep,
    MiniMaxH3DecodeStep,
    MiniMaxH3Ref2VACoreDenoiseStep,
)
from diffusers.modular_pipelines.modular_pipeline import SequentialPipelineBlocks
from diffusers.modular_pipelines.modular_pipeline_utils import OutputParam


def _wire_outputs(num_frames=True):
    """The wire format of the split. `num_frames` is declared by the `ref2va` half alone."""
    return [
        OutputParam.template("prompt_embeds"),
        OutputParam("text_token_tags", description="The per-row modality tag of every row of `prompt_embeds`."),
        OutputParam("height", type_hint=int, description="Resolved height of the generated video in pixels."),
        OutputParam("width", type_hint=int, description="Resolved width of the generated video in pixels."),
        *([OutputParam("num_frames", type_hint=int, description="Resolved number of frames, of the form 17 * n + 5.")]
          if num_frames else []),
    ]


def _generation_outputs():
    """The generation outputs: what the denoising half hands back to the caller."""
    return [
        OutputParam.template("videos"),
        OutputParam("audio", description="The generated soundtrack, of shape (1, 2, num_samples)."),
        OutputParam("sampling_rate", type_hint=int, description="Sample rate of the generated soundtrack in Hz."),
    ]


class MiniMaxH3ConditionerBlocks(SequentialPipelineBlocks):
    """The conditioner half of a split MiniMax-H3: the keyframes on the canvas plus the Qwen3-VL read at layer 50."""

    model_name = "minimax-h3"
    block_classes = [MiniMaxH3AutoBeforeEncodeStep, MiniMaxH3TextEncoderStep]
    block_names = ["before_encode", "text_encoder"]

    @property
    def description(self):
        return (
            "The conditioner half of a split MiniMax-H3 deployment: puts the keyframes onto the target canvas "
            "and encodes MiniMax-H3's presentation of the request into the prompt_embeds / text_token_tags pair "
            "the denoising half consumes. The frame count is the caller's to align."
        )

    @property
    def outputs(self):
        return _wire_outputs(num_frames=False)


class MiniMaxH3GeneratorBlocks(SequentialPipelineBlocks):
    """The denoising half of a split MiniMax-H3: MiniMaxH3Blocks with its text_encoder step removed."""

    model_name = "minimax-h3"
    block_classes = [
        MiniMaxH3AutoBeforeEncodeStep,
        MiniMaxH3AutoVaeEncoderStep,
        MiniMaxH3CoreDenoiseStep,
        MiniMaxH3AfterDenoiseStep,
        MiniMaxH3DecodeStep,
    ]
    block_names = ["before_encode", "vae_encoder", "denoise", "after_denoise", "decode"]

    @property
    def description(self):
        return (
            "The denoising half of a split MiniMax-H3 deployment: the t2va / fl2va branch of MiniMaxH3Blocks "
            "without its text-encoder step, so prompt_embeds and text_token_tags come in as inputs and the "
            "62.14 GiB Qwen3-VL conditioner is never loaded here."
        )

    @property
    def outputs(self):
        return _generation_outputs()


class MiniMaxH3Ref2VAConditionerBlocks(SequentialPipelineBlocks):
    """The conditioner half of a split ref2va: the resolved plan plus the Qwen3-VL read at its 50th layer."""

    model_name = "minimax-h3"
    block_classes = [MiniMaxH3Ref2VASetupStep, MiniMaxH3Ref2VATextEncoderStep]
    block_names = ["setup", "text_encoder"]

    @property
    def description(self):
        return (
            "The conditioner half of a split MiniMax-H3 ref2va deployment: resolves the request plan and encodes "
            "MiniMax-H3's presentation into the prompt_embeds / text_token_tags pair the denoising half consumes."
        )

    @property
    def outputs(self):
        return _wire_outputs()


class MiniMaxH3Ref2VAGeneratorBlocks(SequentialPipelineBlocks):
    """The denoising half of a split ref2va: the ref2va branch with its text_encoder step removed."""

    model_name = "minimax-h3"
    block_classes = [
        MiniMaxH3Ref2VASetupStep,
        MiniMaxH3Ref2VAReferenceEncoderStep,
        MiniMaxH3Ref2VACoreDenoiseStep,
        MiniMaxH3AfterDenoiseStep,
        MiniMaxH3DecodeStep,
    ]
    block_names = ["setup", "reference_encoder", "denoise", "after_denoise", "decode"]

    @property
    def description(self):
        return (
            "The denoising half of a split MiniMax-H3 ref2va deployment: the ref2va branch without its text-encoder "
            "step, so prompt_embeds and text_token_tags come in as inputs and the Qwen3-VL is never loaded here. "
            "The transformer is the transformer_ref partition."
        )

    @property
    def outputs(self):
        return _generation_outputs()
'''
def _ensure_split_blocks():
    """Write the corrected h3_split_blocks.py to WORK_ROOT (idempotent) and put it on sys.path."""
    _SPLIT_BLOCKS_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    needs_write = (not _SPLIT_BLOCKS_LOCAL.exists()
                   or 'MiniMaxH3AutoVaeEncoderStep' not in _SPLIT_BLOCKS_LOCAL.read_text())
    if needs_write:
        _SPLIT_BLOCKS_LOCAL.write_text(_SPLIT_BLOCKS_SOURCE)
    if str(_SPLIT_BLOCKS_LOCAL.parent) not in sys.path:
        sys.path.insert(0, str(_SPLIT_BLOCKS_LOCAL.parent))

# --- Lazy model loaders (per-component, swap between requests) ----------------
_CONDITIONER_PIPE = None
_DENOISER_PIPE = None
_DEVICE = None

def _patch_int8_quant():
    """Patch torchao's Int8WeightOnlyConfig to drop the `version=2` kwarg if not available.

    The released torchao on Colab does not expose `version=`; passing it raises TypeError.
    Without it, int8 tensors aren't pinnable, so `use_stream=False` in group offload.
    """
    try:
        from torchao.quantization import Int8WeightOnlyConfig
        Int8WeightOnlyConfig(version=2)
    except TypeError:
        import torchao.quantization as _tq
        _orig = _tq.Int8WeightOnlyConfig
        class _PatchedInt8(_orig):
            def __init__(self, *args, **kwargs):
                kwargs.pop('version', None)
                super().__init__(*args, **kwargs)
        _tq.Int8WeightOnlyConfig = _PatchedInt8

def _build_offload(device):
    """Standard offload kwargs for INT8 group offload (no stream — pinnable tensors off)."""
    return dict(onload_device=device, offload_device=torch.device('cpu'), use_stream=False)

def _load_conditioner_pipe(verbose=True):
    """Build and load the Qwen3-VL conditioner pipeline (INT8 text_encoder + tiny components).

    Uses `MiniMaxH3ConditionerBlocks` from the upstream split pattern. The pipeline
    only declares the components the text_encoder + keyframe-encoder steps need
    (text_encoder, tokenizer, processor, vae, audio_vae, image_processor) — the
    transformer is NEVER loaded by this pipeline, so peak host RAM ≈ 30 GB INT8.
    """
    global _CONDITIONER_PIPE
    if _CONDITIONER_PIPE is not None:
        return _CONDITIONER_PIPE

    # If the denoiser is cached, free it first — both can't coexist in 53 GB host RAM.
    global _DENOISER_PIPE
    if _DENOISER_PIPE is not None:
        if verbose:
            print('  [cond] Freeing cached denoiser to make room for text_encoder (~30 GB INT8)...')
        _DENOISER_PIPE = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    _ensure_split_blocks()
    from h3_split_blocks import MiniMaxH3ConditionerBlocks
    from diffusers import ModularPipeline, TorchAoConfig
    from diffusers.modular_pipelines.components_manager import ComponentsManager
    from transformers import Qwen3VLForConditionalGeneration
    from transformers import TorchAoConfig as TransformersTorchAoConfig
    from torchao.quantization import Int8WeightOnlyConfig

    _patch_int8_quant()

    manager = ComponentsManager()
    blocks = MiniMaxH3ConditionerBlocks()
    if verbose:
        print(f'  [cond] Building conditioner pipeline from {MODEL_REPO} ...')
    pipe = blocks.init_pipeline(MODEL_REPO, components_manager=manager)

    # Load the INT8 text_encoder directly from the Hub. meta-init model + num_workers=1
    # keeps peak host RAM at one shard (~4 GB bf16) + accumulating INT8 (~30 GB).
    text_encoder = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_REPO, subfolder='text_encoder', dtype=torch.bfloat16,
        quantization_config=TransformersTorchAoConfig(
            Int8WeightOnlyConfig(),
            modules_to_not_convert=_TEXT_ENCODER_KEEP_BF16,
        ),
        device_map='cpu',
        max_memory=MAX_MEMORY,
        cache_dir=os.environ.get('HF_HOME'),
    )
    pipe.update_components(text_encoder=text_encoder)

    # Load the remaining tiny components from the cached checkpoint.
    pipe.load_components(dtype=torch.bfloat16)

    # The conditioner blocks (`MiniMaxH3AutoBeforeEncodeStep` + `MiniMaxH3TextEncoderStep`)
    # don't declare the VAE — keyframe VAE encoding lives in the *denoiser* half
    # (`MiniMaxH3GeneratorBlocks.MiniMaxH3AutoVaeEncoderStep`). So `pipe.vae` is not an
    # attribute here; the denoiser will load it on its own. Skip the .to(_DEVICE) call.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if verbose:
        free, total = torch.cuda.mem_get_info()
        print(f'  [cond] VRAM after conditioner load: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')

    _CONDITIONER_PIPE = pipe
    return pipe

def _load_denoiser_pipe(verbose=True):
    """Build and load the denoiser pipeline (INT8 transformer + tiny components).

    Uses `MiniMaxH3GeneratorBlocks` from the upstream split pattern. The pipeline
    only declares the components the denoise + decode steps need (transformer,
    scheduler, audio_scheduler, vae, audio_vae, video_processor) — text_encoder is
    NEVER loaded. Peak host RAM ≈ 30 GB INT8.
    """
    global _DENOISER_PIPE
    if _DENOISER_PIPE is not None:
        return _DENOISER_PIPE

    # If the conditioner is cached, free it first — both can't coexist in 53 GB host RAM.
    global _CONDITIONER_PIPE
    if _CONDITIONER_PIPE is not None:
        if verbose:
            print('  [gen] Freeing cached conditioner to make room for transformer (~30 GB INT8)...')
        _CONDITIONER_PIPE = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    _ensure_split_blocks()
    from h3_split_blocks import MiniMaxH3GeneratorBlocks
    from diffusers import ModularPipeline, MiniMaxH3Transformer3DModel, TorchAoConfig
    from diffusers.modular_pipelines.components_manager import ComponentsManager
    from torchao.quantization import Int8WeightOnlyConfig

    _patch_int8_quant()

    manager = ComponentsManager()
    blocks = MiniMaxH3GeneratorBlocks()
    if verbose:
        print(f'  [gen] Building denoiser pipeline from {MODEL_REPO} ...')
    pipe = blocks.init_pipeline(MODEL_REPO, components_manager=manager, collection='h3')

    # Load the INT8 transformer directly from the Hub.
    transformer = MiniMaxH3Transformer3DModel.from_pretrained(
        MODEL_REPO, subfolder='transformer', dtype=torch.bfloat16,
        quantization_config=TorchAoConfig(
            Int8WeightOnlyConfig(),
            modules_to_not_convert=_TRANSFORMER_KEEP_BF16,
        ),
        device_map='cpu',
        max_memory=MAX_MEMORY,
        cache_dir=os.environ.get('HF_HOME'),
    )
    pipe.update_components(transformer=transformer)

    # Load the remaining tiny components.
    pipe.load_components(dtype=torch.bfloat16)

    # Lower the minimum duration to 2 s. The released MiniMax-H3 generates
    # 5-15 s (a hard floor of `17 * n + 5` latent-aligned frames), but the
    # upstream Space's reference implementation also handles 2-5 s on the
    # released weights by lowering `MiniMaxH3ModularPipeline.min_duration`
    # to 2.0. On the L4 with int8-prequant-split this matters: a 56-frame
    # clip has ~4x less activation memory than the default 124-frame clip,
    # which is the difference between OOMing and finishing on 22 GB.
    from diffusers.modular_pipelines.minimax_h3.modular_pipeline import MiniMaxH3ModularPipeline as _H3P
    _H3P.min_duration = property(lambda self: 2.0)

    # VAEs must stay on CPU during the denoise loop — they are only needed at the
    # final decode step, and a bf16 video VAE eats ~10 GB of VRAM. With the L4's
    # 22 GB, the activations of a 124-frame packed RoPE forward already reach
    # 18-20 GB; pinning the VAEs on top blows the OOM budget.
    #
    # To make the offload hooks actually fire on `vae.decode(...)` (which is what
    # MiniMaxH3DecodeStep calls directly), we wrap each VAE's `decode` method with
    # a hook that calls `pre_forward` first. Without this, the VAE stays on host
    # when the latents arrive on the card and the decode raises. (Same pattern as
    # the upstream `multimodalart/minimax-h3` Space's `_arm_decode_hooks`.)
    def _arm_vae_decode_hook(vae):
        inner = vae.decode
        def armed(*args, _vae=vae, _decode=inner, **kwargs):
            hook = getattr(_vae, '_hf_hook', None)
            if hook is not None:
                hook.pre_forward(_vae)
            return _decode(*args, **kwargs)
        vae.decode = armed
    for _name in ('vae', 'audio_vae'):
        _mod = getattr(pipe, _name, None)
        if _mod is not None and hasattr(_mod, '_hf_hook'):
            _arm_vae_decode_hook(_mod)

    # Stream transformer blocks from CPU. Skip `_native_cudnn` on the L4 — it
    # holds the full rotary buffer pinned on GPU and the activation footprint
    # at the RoPE layer is the failure point. The SDPA default uses less
    # VRAM at a ~10% throughput cost; that's the right trade on 22 GB.
    offload = _build_offload(_DEVICE)
    pipe.transformer.enable_group_offload(
        offload_type='block_level', num_blocks_per_group=1, **offload,
    )
    pipe.transformer.requires_grad_(False)
    if verbose:
        free, total = torch.cuda.mem_get_info()
        print(f'  [gen] VRAM after denoiser load: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')

    _DENOISER_PIPE = pipe
    return pipe

def _load_sdnq_pipe(verbose=True):
    """SDNQ prequantized checkpoint path — the recommended L4 + 53 GB host RAM recipe.

    The SDNQ checkpoint at `OzzyGT/MiniMax_H3_sdnq_dynamic_{4bit,8bit}` is in diffusers
    format with the quantization_config baked into the per-shard safetensors keys. SDNQ
    registers itself with diffusers' quantizer registry, so loading goes through
    diffusers' quantizer hooks — no `quantization_config=` argument needed.

    The 4-bit checkpoint is 37 GB on disk (transformer 17.5 GB + text_encoder
    19.6 GB). Loading it straight into VRAM OOMs the L4: the Qwen3-VL text encoder's
    '_keep_in_fp32_modules' ('model.visual', 'model.language_model.embed_tokens',
    `model.language_model.norm`, `lm_head`) alone weigh ~9 GB in bf16, and pinning
    the whole text encoder + the bf16 video VAE (~10 GB) + audio VAE (~0.5 GB)
    adds up to ~22 GB on a 22 GB card. The diffusers `ComponentsManager`
    auto-cpu-offload solves this: it registers a hook on each component that
    onloads to GPU at the start of its forward and offloads back to CPU at the end,
    so we never have more than ~10-12 GB resident on the L4 at once. The 8-bit
    variant (94 GB on disk) needs a host with ≥100 GB host RAM and is not the
    default L4 recipe.

    We bypass `ModularPipeline.from_pretrained(MODEL_REPO, workflow=...)` because
    the SDNQ repo's `modular_model_index.json` has no `auto_map` (the blocks
    class lives in diffusers, not in the hub repo), and the default loader
    raises `KeyError: 'auto_map'` on it. Building the pipeline directly —
    `MiniMaxH3ModularPipeline(blocks=MiniMaxH3Blocks(), pretrained_model_name_or_path=MODEL_REPO, ...)`
    — gives us the same outcome and runs cleanly.
    """
    global _DENOISER_PIPE
    if _DENOISER_PIPE is not None:
        return _DENOISER_PIPE

    # Make sure the sdnq package is importable so diffusers' quantizer registry sees it.
    try:
        import sdnq  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sdnq>=0.2.3'], check=False)
        import sdnq  # noqa: F401
    # Tell diffusers' transformers integration to register SDNQ for the text encoder load.
    os.environ.setdefault('DIFFUSERS_SDNQ_TRANSFORMERS', '1')

    from diffusers import MiniMaxH3Blocks, MiniMaxH3ModularPipeline
    from diffusers.modular_pipelines.components_manager import ComponentsManager

    manager = ComponentsManager()
    if verbose:
        print(f'  [gen] Loading SDNQ pipeline from {MODEL_REPO} (workflow=fl2va) ...')
    blocks = MiniMaxH3Blocks()
    pipe = MiniMaxH3ModularPipeline(
        blocks=blocks,
        pretrained_model_name_or_path=MODEL_REPO,
        components_manager=manager,
    )
    pipe.load_components(workflow='fl2va', dtype=torch.bfloat16)

    # Block-level group offload for the transformer (60+ blocks, INT4/INT8 weights).
    # VAEs and the text encoder are managed by the ComponentsManager's auto-cpu-offload:
    # the manager registers a hook on each component that onloads it to GPU at the
    # start of its forward and offloads it back to CPU after, so we never have
    # more than one or two of {text_encoder, vae, audio_vae} resident on the
    # 22 GB L4 at once. The manager picked the order based on per-step memory
    # profiles; with a 3 GB reserve it can keep VAEs on GPU during the denoise
    # step and have room for the transformer's streaming block.
    manager.enable_auto_cpu_offload(device=_DEVICE, memory_reserve_margin='3GB')
    try:
        pipe.transformer.set_attention_backend('_native_cudnn')
    except Exception:
        pass
    offload = dict(onload_device=_DEVICE, offload_device=torch.device('cpu'), use_stream=True)
    pipe.transformer.enable_group_offload(
        offload_type='block_level', num_blocks_per_group=1, **offload,
    )
    if verbose:
        free, total = torch.cuda.mem_get_info()
        print(f'  [gen] VRAM after setup: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')

    _DENOISER_PIPE = pipe
    return pipe

def _load_bf16_pipe(verbose=True):
    """Single-pipeline bf16 + auto_cpu_offload path for A100 80GB. No quantization."""
    global _DENOISER_PIPE
    if _DENOISER_PIPE is not None:
        return _DENOISER_PIPE
    from diffusers import MiniMaxH3Blocks, MiniMaxH3ModularPipeline
    from diffusers.modular_pipelines.components_manager import ComponentsManager

    manager = ComponentsManager()
    if verbose:
        print(f'  [gen] Loading bf16 pipeline from {MODEL_REPO} (workflow=fl2va) ...')
    # Bypass 'ModularPipeline.from_pretrained' — the hub repo's modular_model_index.json
    # has no 'auto_map' (the blocks class lives in diffusers, not in the hub), and the
    # default loader would either raise 'KeyError: 'auto_map'' or fall back to a bare
    # 'SequentialPipelineBlocks' instance without the '_workflow_map' declared on
    # 'MiniMaxH3Blocks'. Building the pipeline directly avoids both failure modes.
    pipe = MiniMaxH3ModularPipeline(
        blocks=MiniMaxH3Blocks(),
        pretrained_model_name_or_path=MODEL_REPO,
        components_manager=manager,
    )
    pipe.load_components(workflow='fl2va', dtype=torch.bfloat16)
    from diffusers.modular_pipelines.minimax_h3.modular_pipeline import MiniMaxH3ModularPipeline as _H3P
    _H3P.min_duration = property(lambda self: 2.0)
    try:
        pipe.transformer.set_attention_backend('_flash_3_hub')
    except Exception:
        pipe.transformer.set_attention_backend('_native_cudnn')
    manager.enable_auto_cpu_offload(device=str(_DEVICE), memory_reserve_margin='12GB')
    if verbose:
        free, total = torch.cuda.mem_get_info()
        print(f'  [gen] VRAM after setup: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')
    _DENOISER_PIPE = pipe
    return pipe

def free_minimax_h3():
    """Unload the model(s) and free GPU + host memory."""
    global _CONDITIONER_PIPE, _DENOISER_PIPE, _DEVICE
    for var in ('_CONDITIONER_PIPE', '_DENOISER_PIPE'):
        if globals().get(var) is not None:
            del globals()[var]
    _DEVICE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- Full generation pipeline ─────────────────────────────────────────
def _save_conditioner_state(state, num_frames):
    """Persist a conditioner's intermediate state to a temp safetensors file."""
    from safetensors.torch import save_file
    path = WORK_ROOT / f'cond_state_{int(time.time()*1000)}.safetensors'
    payload = {
        'prompt_embeds': state.get('prompt_embeds').detach().cpu().contiguous(),
        'text_token_tags': state.get('text_token_tags').detach().cpu().contiguous(),
    }
    if state.get('condition_latents') is not None:
        # FL2VA: encode keyframe condition latents as a single concatenated tensor.
        conds = state.get('condition_latents')
        for i, c in enumerate(conds):
            payload[f'condition_latents.{i}'] = c.detach().cpu().contiguous()
        payload['condition_latents.count'] = torch.tensor([len(conds)], dtype=torch.long)
    if state.get('height') is not None:
        payload['height'] = torch.tensor([int(state.get('height'))], dtype=torch.long)
    if state.get('width') is not None:
        payload['width'] = torch.tensor([int(state.get('width'))], dtype=torch.long)
    payload['num_frames'] = torch.tensor([int(num_frames)], dtype=torch.long)
    save_file(payload, str(path), metadata={'format': 'pt'})
    return path

def _load_conditioner_state(path):
    """Inverse of _save_conditioner_state. Returns a dict of tensors / scalars."""
    from safetensors import safe_open
    state = {}
    with safe_open(str(path), framework='pt') as f:
        for k in f.keys():
            state[k] = f.get_tensor(k)
    return state

def _resolve_prompt_embeds_remote(prompt, image_path=None, last_image_path=None,
                                   canvas=DEFAULT_CANVAS, num_frames=124,
                                   rewrite_prompt=False, verbose=True):
    """Call the upstream `multimodalart/qwen3vl-conditioner` HF Space for prompt encoding.

    Returns a state dict identical to what the local conditioner would produce, so the
    denoiser pipeline can consume it the same way.
    """
    from gradio_client import Client, handle_file
    from safetensors import safe_open

    if verbose:
        print(f'  [cond] Connecting to {CONDITIONER_SPACE} ...')
    t0 = time.time()
    client = Client(CONDITIONER_SPACE)
    path, plan = client.predict(
        prompt=prompt,
        image_path=handle_file(image_path) if image_path else None,
        last_image_path=handle_file(last_image_path) if last_image_path else None,
        canvas=canvas,
        num_frames=int(num_frames),
        rewrite_prompt=bool(rewrite_prompt),
        api_name='/encode',
    )
    if verbose:
        print(f'  [cond] Connected and encoded in {time.time()-t0:.1f}s ({plan.get("num_text_tokens", "?")} tokens)')
    with safe_open(path, framework='pt') as handle:
        meta = handle.metadata() or {}
        state = {
            'prompt_embeds': handle.get_tensor('prompt_embeds').cpu().contiguous(),
            'text_token_tags': handle.get_tensor('text_token_tags').cpu().contiguous(),
            'height': int(meta.get('height', 544)),
            'width': int(meta.get('width', 960)),
            'num_frames': int(meta.get('num_frames', num_frames)),
            'plan': plan,
        }
    return state

def generate_video(prompt, image_path=None, last_image_path=None,
                     canvas=DEFAULT_CANVAS, duration=5, steps=28, seed=42,
                     rewrite_prompt=False, verbose=True):
    global _CONDITIONER_PIPE, _DENOISER_PIPE
    """Generate one video. Returns (out_path, report).

    The path through `RECIPE` (resolved from `LOAD_MODE` in STEP 1):
      'bf16-a100'           : load single bf16 pipeline → call once → done.
      'int8-prequant-split' : load conditioner → run → save state → free →
                              load denoiser → run with state → free.
      'int8-remote-text'    : call remote HF Space conditioner → load denoiser →
                              run with remote state → free.
    """
    from PIL import Image, ImageOps
    from diffusers.utils import encode_video

    global _DEVICE
    _DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_frames = snap_frames(duration)

    def keyframe(path):
        return ImageOps.exif_transpose(Image.open(path)).convert('RGB') if path else None

    denoise_started = time.time()

    if RECIPE == 'bf16-a100':
        pipe = _load_bf16_pipe(verbose=verbose)
        state = pipe(
            prompt=prompt,
            image=keyframe(image_path),
            last_image=keyframe(last_image_path),
            num_frames=num_frames,
            num_inference_steps=int(steps),
            generator=torch.Generator('cpu').manual_seed(int(seed)),
            output=['videos', 'audio', 'sampling_rate'],
        )
        report_plan = {}
    elif RECIPE in ('sdnq-4bit', 'sdnq-8bit'):
        # SDNQ prequantized checkpoint — single pipeline, full fl2va flow. The text
        # encoder and transformer are both resident (4-bit: 37 GB on disk / ~37 GB
        # resident; 8-bit: 94 GB on disk / ~94 GB resident).
        pipe = _load_sdnq_pipe(verbose=verbose)
        state = pipe(
            prompt=prompt,
            image=keyframe(image_path),
            last_image=keyframe(last_image_path),
            num_frames=num_frames,
            num_inference_steps=int(steps),
            generator=torch.Generator('cpu').manual_seed(int(seed)),
            output=['videos', 'audio', 'sampling_rate'],
        )
        report_plan = {}
    elif RECIPE == 'int8-prequant-split':
        # 1. Conditioner: text_encoder + keyframe encode
        if verbose:
            print(f'\n  Phase 1: Conditioning (INT8 Qwen3-VL on CPU) ...')
        cond_pipe = _load_conditioner_pipe(verbose=verbose)
        cond_state = cond_pipe(
            prompt=prompt,
            image=keyframe(image_path),
            last_image=keyframe(last_image_path),
            num_frames=num_frames,
        )
        cond_state_path = _save_conditioner_state(cond_state, num_frames)
        height = int(cond_state.get('height') or 544)
        width = int(cond_state.get('width') or 960)
        actual_num_frames = int(cond_state.get('num_frames') or num_frames)
        report_plan = {}
        # 2. Free the text_encoder before loading the transformer
        del _CONDITIONER_PIPE
        _CONDITIONER_PIPE = None
        del cond_pipe
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # 3. Load the denoiser
        if verbose:
            print(f'\n  Phase 2: Loading INT8 denoiser (transformer on CPU) ...')
        denoise_pipe = _load_denoiser_pipe(verbose=verbose)
        # 4. Restore state and run
        from safetensors import safe_open
        with safe_open(str(cond_state_path), framework='pt') as f:
            prompt_embeds = f.get_tensor('prompt_embeds')
            text_token_tags = f.get_tensor('text_token_tags')
        if verbose:
            print(f'  Phase 3: Denoising {steps} steps at {width}x{height}, {actual_num_frames} frames ...')
        state = denoise_pipe(
            prompt_embeds=prompt_embeds,
            text_token_tags=text_token_tags,
            image=keyframe(image_path),
            last_image=keyframe(last_image_path),
            height=height,
            width=width,
            num_frames=actual_num_frames,
            num_inference_steps=int(steps),
            generator=torch.Generator('cpu').manual_seed(int(seed)),
            output=['videos', 'audio', 'sampling_rate'],
        )
        cond_state_path.unlink(missing_ok=True)
    elif RECIPE == 'int8-remote-text':
        # 1. Get prompt embeds from the upstream Space
        if verbose:
            print(f'\n  Phase 1: Remote conditioning on {CONDITIONER_SPACE} ...')
        cond_state = _resolve_prompt_embeds_remote(
            prompt, image_path, last_image_path, canvas, num_frames,
            rewrite_prompt, verbose=verbose,
        )
        height = cond_state['height']
        width = cond_state['width']
        actual_num_frames = cond_state['num_frames']
        report_plan = cond_state.get('plan') or {}
        # 2. Load the local denoiser and run
        if verbose:
            print(f'\n  Phase 2: Loading INT8 denoiser (transformer on CPU) ...')
        denoise_pipe = _load_denoiser_pipe(verbose=verbose)
        if verbose:
            print(f'  Phase 3: Denoising {steps} steps at {width}x{height}, {actual_num_frames} frames ...')
        state = denoise_pipe(
            prompt_embeds=cond_state['prompt_embeds'],
            text_token_tags=cond_state['text_token_tags'],
            image=keyframe(image_path),
            last_image=keyframe(last_image_path),
            height=height,
            width=width,
            num_frames=actual_num_frames,
            num_inference_steps=int(steps),
            generator=torch.Generator('cpu').manual_seed(int(seed)),
            output=['videos', 'audio', 'sampling_rate'],
        )
    else:
        raise RuntimeError(f'Unknown RECIPE: {RECIPE}')

    generate_seconds = time.time() - denoise_started
    if verbose:
        print(f'  Generation done in {generate_seconds:.0f}s')

    # Free VRAM before muxing (muxing is CPU-side anyway).
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Mux
    videos = state['videos']
    audio = state['audio']
    sampling_rate = state['sampling_rate']
    frames = videos[0]
    audio_data = audio[0].cpu() if hasattr(audio[0], 'cpu') else audio[0]
    sr = sampling_rate if sampling_rate is not None else 32000

    out_dir = OUT_DIR / f'gen_{int(time.time())}'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = str(out_dir / 'output.mp4')
    encode_video(frames, fps=FPS, output_path=out_path, audio=audio_data, audio_sample_rate=sr)
    if verbose:
        sz = os.path.getsize(out_path) / 1024 / 1024
        print(f'  Output: {out_path} ({sz:.1f} MB)')

    return out_path, {
        'width': width,
        'height': height,
        'num_frames': actual_num_frames,
        'steps': int(steps),
        'seed': int(seed),
        'generate_seconds': generate_seconds,
        'recipe': RECIPE,
        'conditioner_tokens': report_plan.get('num_text_tokens', 0),
    }

def free_cuda(verbose=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            free, total = torch.cuda.mem_get_info()
            print(f'  [cuda] free={free/1024**3:.1f} GB / total={total/1024**3:.1f} GB')

print('STEP 3 complete — official MiniMax-H3 pipeline (PR #14355) loaded.')
print('Next: run STEP 4 to open the Gradio UI.')


In [ ]:
#@title STEP 4 — Gradio UI (text-to-video + image-to-video with audio)
"""
• Two-column layout: left = controls, right = video output
• Prompt + optional first/last frame images
• Canvas selector (aspect ratios)
• Duration slider (5-14 seconds)
• Steps slider (10-40)
• Seed (0 or negative = random)
• Prompt rewrite toggle
"""
import os, sys, time, pathlib, traceback, random
import torch
import gradio as gr

CSS = """
#col-container   { margin: 0 auto; max-width: 1400px; }
#main-title h1   { font-size: 2.4em !important; }
"""

with gr.Blocks(css=CSS, delete_cache=(600, 600)) as demo:
    gr.Markdown(
        '# **MiniMax-H3 — Video + Audio Generation**',
        elem_id='main-title',
    )
    gr.Markdown(
        'Generate video with synchronized audio from a text prompt '
        '(optionally with first/last frame images). '
        'Powered by [MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) (33B, INT8 quantized).'
    )

    with gr.Row(elem_id='col-container'):
        with gr.Column(scale=1, min_width=380):
            prompt = gr.Textbox(
                label='Prompt',
                lines=3,
                value='A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot',
                info='Describe the scene. Audio (ambience, foley, speech) is generated automatically.',
            )
            with gr.Row():
                image = gr.Image(label='First frame (optional)', type='filepath', height=200)
                last_image = gr.Image(label='Last frame (optional)', type='filepath', height=200)
            btn_generate = gr.Button('Generate video', variant='primary')
            with gr.Accordion('Advanced options', open=False):
                canvas = gr.Dropdown(
                    label='Canvas (aspect ratio)',
                    choices=list(CANVASES.keys()),
                    value=DEFAULT_CANVAS,
                    info='Resolution and aspect ratio. "fast" variants are smaller and quicker.',
                )
                duration = gr.Slider(
                    MIN_UI_DURATION, MAX_UI_DURATION, value=5, step=1,
                    label='Duration (seconds)',
                    info='Snapped to the nearest valid frame count (17*n+5 at 24 fps). 2-5s clips fit on L4; 5-15s need 40+ GB VRAM.',
                )
                steps = gr.Slider(
                    10, 40, value=28, step=1,
                    label='Inference steps',
                    info='More steps = higher quality but slower. 28 is the default.',
                )
                seed = gr.Number(
                    value=42,
                    label='Seed (0 or negative = random)',
                    info='Different seeds = different videos. Use 0 or negative for a random seed each run.',
                    precision=0,
                )
                rewrite_prompt = gr.Checkbox(
                    value=False,
                    label='Rewrite prompt',
                    info='Hand the prompt to the pipeline unchanged; this is a placeholder for future prompt refinement.',
                )
            status_box = gr.Textbox(
                label='Status', interactive=False, lines=3,
                placeholder='Awaiting generation...',
            )

        with gr.Column(scale=2):
            output_video = gr.Video(label='Video + soundtrack', height=500)
            report_box = gr.Markdown(visible=False)
            with gr.Accordion('Downloads', open=False):
                dl_video = gr.File(label='Download .mp4')

    # --- Event wiring ---
    def generate(prompt_text, img_path, last_img_path, canvas_label,
                  dur, n_steps, s, rewrite, progress=gr.Progress(track_tqdm=True)):
        try:
            if not prompt_text or not prompt_text.strip():
                raise gr.Error('A prompt is required.')
            # Seed: 0 or negative = random
            actual_seed = int(s)
            if actual_seed <= 0:
                actual_seed = random.randint(1, 2**31 - 1)
            progress(0.0, desc='Conditioning ...')
            out_path, report = generate_video(
                prompt=prompt_text,
                image_path=img_path if img_path else None,
                last_image_path=last_img_path if last_img_path else None,
                canvas=canvas_label,
                duration=dur,
                steps=n_steps,
                seed=actual_seed,
                rewrite_prompt=rewrite,
                verbose=True,
            )
            report_md = (
                f'`{report["width"]}x{report["height"]}`, {report["num_frames"]} frames '
                f'({report["num_frames"]/FPS:.1f}s), {report["steps"]} steps · '
                f'denoise {report["generate_seconds"]:.0f}s · seed {report["seed"]}'
            )
            if report.get('refined_prompt'):
                report_md += f'\n\n**Refined prompt:** {report["refined_prompt"]}'
            free_cuda()
            return (
                gr.update(value=out_path),
                gr.update(value=report_md, visible=True),
                gr.update(value=out_path),
            )
        except Exception as e:
            traceback.print_exc(limit=4)
            raise gr.Error(f'Generation failed: {e}')

    btn_generate.click(
        generate,
        inputs=[prompt, image, last_image, canvas, duration, steps, seed, rewrite_prompt],
        outputs=[output_video, report_box, dl_video],
    )

    def _welcome():
        return (
            'Enter a prompt, optionally upload first/last frame images, '
            'then click "Generate video". Use seed 0 for random. '
            'First run downloads weights (~134 GB) then loads + quantizes the model '
            '(~5-10 min after STEP 2).'
        )
    demo.load(_welcome, inputs=None, outputs=[status_box])

# --- Queue + launch ────────────────────────────────────────────────────
demo.queue(concurrency_limit=2, max_size=4)
try:
    from IPython.display import clear_output
    clear_output()
    clear_output(wait=True)
except Exception:
    pass
demo.launch(share=False, server_name='0.0.0.0', server_port=7860, show_error=True, height=1100)


In [ ]:
#@title STEP 5 — Keep alive + session summary
"""Standard AEI-suite keep-alive cell."""
import os, sys, time, pathlib
import IPython
from IPython.display import display, Javascript

print('='*72)
print('Keep-alive timer started.')
print('='*72)

try:
    summary = {
        'cache_root'    : str(drive_root),
        'ckpt_dir'      : str(CKPT_DIR),
        'out_dir'       : str(OUT_DIR),
        'torch'         : torch.__version__,
        'cuda'          : torch.version.cuda,
        'diffusers'     : diffusers.__version__,
        'transformers'  : transformers.__version__,
        'gpu'           : None,
    }
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        summary['gpu'] = f'{p.name}  ({p.total_memory / (1024**3):.1f} GB)'
    print('\n  Session summary')
    print('  ' + '-'*68)
    for k, v in summary.items():
        print(f'  {k:18s}: {v}')
except Exception as e:
    print(f'  WARN: summary print failed: {e}')

display(Javascript('''
function ClickConnect() {
  console.log("Keeping Colab alive — ", new Date().toLocaleTimeString());
  document.querySelector("colab-connect-button")?.click();
}
setInterval(ClickConnect, 60000);
'''))
print('\n  Keep-alive timer registered (60 s interval).')


In [ ]:
#@title STEP 6 — Quick test (single video generation)
"""Stand-alone test. Generates one video with default parameters."""
import os, sys, time, pathlib, random
from IPython.display import display, FileLink

print('='*72)
print('MiniMax-H3 — single-video quick test')
print('='*72)

PROMPT = 'A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot'  #@param {type:"string"}
IMAGE_PATH = ''  #@param {type:"string"}
LAST_IMAGE_PATH = ''  #@param {type:"string"}
CANVAS = "960x544 · 16:9 fast"  #@param ['960x544 · 16:9 fast', '1024x576 · 16:9 fast', '1344x768 · 16:9 full', '544x960 · 9:16 fast', '640x1152 · 9:16', '544x544 · 1:1 fast', '768x768 · 1:1 full']
DURATION = 5  #@param {type:"slider", min:2, max:14, step:1}
STEPS = 28  #@param {type:"slider", min:10, max:40, step:1}
SEED = 42  #@param {type:"integer"}
REWRITE = False  #@param {type:"boolean"}

# Seed: 0 or negative = random
if SEED <= 0:
    SEED = random.randint(1, 2**31 - 1)
    print(f'  Random seed: {SEED}')

img_path = IMAGE_PATH.strip() if IMAGE_PATH.strip() else None
last_img_path = LAST_IMAGE_PATH.strip() if LAST_IMAGE_PATH.strip() else None

print(f'  Prompt      : {PROMPT[:60]}...')
print(f'  First frame : {img_path or "(none)"}')
print(f'  Last frame  : {last_img_path or "(none)"}')
print(f'  Canvas      : {CANVAS}')
print(f'  Duration    : {DURATION}s')
print(f'  Steps       : {STEPS}')
print(f'  Seed        : {SEED}')
print()

t0 = time.time()
out_path, report = generate_video(
    prompt=PROMPT,
    image_path=img_path,
    last_image_path=last_img_path,
    canvas=CANVAS,
    duration=DURATION,
    steps=STEPS,
    seed=SEED,
    rewrite_prompt=REWRITE,
    verbose=True,
)
total_time = time.time() - t0

print()
print('='*72)
print(f'Video generated in {total_time:.0f}s')
print(f'  Resolution : {report["width"]}x{report["height"]}')
print(f'  Frames     : {report["num_frames"]} ({report["num_frames"]/FPS:.1f}s)')
print(f'  Denoise    : {report["generate_seconds"]:.0f}s')
print(f'  Seed       : {report["seed"]}')
print(f'  Output     : {out_path}')
sz = os.path.getsize(out_path) / 1024 / 1024
print(f'  Size       : {sz:.1f} MB')
print('='*72)

display(FileLink(out_path, result_html_prefix='Download video: '))
free_cuda()
print('\nSTEP 6 complete. Open the Gradio UI (STEP 4) for the full experience.')


In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list
"""
Advanced batch processor. Reads a JSON file containing a list of scenes,
each with its own prompt, optional keyframe images, canvas, duration,
steps, and seed. The model is loaded once and reused.

JSON format (a list of objects):
```json
[
  {
    "prompt": "A red fox trotting through a snowy pine forest at dawn",
    "image": "/content/drive/MyDrive/keyframes/fox_start.png",
    "last_image": "/content/drive/MyDrive/keyframes/fox_end.png",
    "canvas": "960x544 · 16:9 fast",
    "duration": 6,
    "steps": 28,
    "seed": 42
  },
  {
    "prompt": "A busy night market, neon signs reflecting in puddles",
    "canvas": "544x960 · 9:16 fast",
    "duration": 5,
    "seed": 0
  }
]
```

Fields (all optional except `prompt`):
  - prompt:      (required) text description of the scene
  - image:       (optional) path to first frame image
  - last_image:  (optional) path to last frame image
  - canvas:      (optional, default DEFAULT_CANVAS) one of the CANVASES keys
  - duration:    (optional, default 5) seconds (5-14)
  - steps:       (optional, default 28) inference steps (10-40)
  - seed:        (optional, default 0 = random) 0 or negative = random
  - rewrite:     (optional, default false) rewrite prompt via conditioner

A progress log is written to batch_log.jsonl for resume after disconnect.
"""
import os, sys, time, json, pathlib, traceback, random
import torch
from IPython.display import display, FileLink

print('='*72)
print('MiniMax-H3 — Batch generation (JSON scene list)')
print('='*72)

BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/MiniMax-H3/batch_scenes.json'  #@param {type:"string"}
DEFAULT_DURATION = 5  #@param {type:"slider", min:5, max:14, step:1}
DEFAULT_STEPS = 28  #@param {type:"slider", min:10, max:40, step:1}
DEFAULT_CANVAS = "960x544 · 16:9 fast"  #@param ['960x544 · 16:9 fast', '1024x576 · 16:9 fast', '1344x768 · 16:9 full', '544x960 · 9:16 fast', '640x1152 · 9:16', '768x1344 · 9:16 full', '544x544 · 1:1 fast', '768x768 · 1:1 full', '768x576 · 4:3 fast', '1024x768 · 4:3 full', '576x768 · 3:4 fast', '768x1024 · 3:4 full', '1152x512 · 21:9 fast', '1536x672 · 21:9 full']
SKIP_EXISTING = True  #@param {type:"boolean"}
REWRITE_DEFAULT = False  #@param {type:"boolean"}

# --- Load and validate the JSON scene list ────────────────────────────
json_path = pathlib.Path(BATCH_JSON_PATH)
if not json_path.exists():
    # Create a template if the file doesn't exist
    json_path.parent.mkdir(parents=True, exist_ok=True)
    template = [
        {"prompt": "A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot",
         "canvas": DEFAULT_CANVAS, "duration": DEFAULT_DURATION, "steps": DEFAULT_STEPS, "seed": 42},
        {"prompt": "A busy night market, neon signs reflecting in puddles, sizzling street food",
         "canvas": "544x960 · 9:16 fast", "duration": 5, "steps": 28, "seed": 0},
    ]
    json_path.write_text(json.dumps(template, indent=2, ensure_ascii=False))
    print(f'  [INFO] Template created at {json_path}')
    print(f'  [INFO] Edit it with your scenes, then re-run this cell.')
    raise SystemExit(0)

try:
    scenes = json.loads(json_path.read_text())
except json.JSONDecodeError as e:
    raise SystemExit(f'[ERROR] Invalid JSON in {json_path}: {e}')

if not isinstance(scenes, list):
    raise SystemExit(f'[ERROR] JSON must be a list of scene objects, got {type(scenes).__name__}')

# Validate scenes
valid_scenes = []
for i, scene in enumerate(scenes):
    if not isinstance(scene, dict):
        print(f'  [WARN] Scene {i}: not a dict, skipping')
        continue
    prompt = scene.get('prompt', '').strip()
    if not prompt:
        print(f'  [WARN] Scene {i}: empty prompt, skipping')
        continue
    # Apply defaults
    scene.setdefault('canvas', DEFAULT_CANVAS)
    scene.setdefault('duration', DEFAULT_DURATION)
    scene.setdefault('steps', DEFAULT_STEPS)
    scene.setdefault('seed', 0)
    scene.setdefault('rewrite', REWRITE_DEFAULT)
    # Validate canvas
    if scene['canvas'] not in CANVASES:
        print(f'  [WARN] Scene {i}: canvas not found, using default')
        scene['canvas'] = DEFAULT_CANVAS
    # Validate image paths
    for key in ('image', 'last_image'):
        val = scene.get(key, '')
        if val:
            p = pathlib.Path(val)
            if not p.exists():
                print(f'  [WARN] Scene {i}: {key} "{val}" not found')
                scene[key] = None
            else:
                scene[key] = str(p)
        else:
            scene[key] = None
    valid_scenes.append(scene)

print(f'  JSON file   : {json_path}')
print(f'  Scenes      : {len(valid_scenes)} (of {len(scenes)} parsed)')
print(f'  Defaults    : canvas={DEFAULT_CANVAS}, duration={DEFAULT_DURATION}s, steps={DEFAULT_STEPS}')
print()

# --- Output directory ────────────────────────────────────────────────
output_subdir = OUT_DIR / f'batch_{int(time.time())}'
output_subdir.mkdir(parents=True, exist_ok=True)
batch_log = output_subdir / 'batch_log.jsonl'

# --- Resume support: check which scenes are already done ─────────────
already_done = set()
if SKIP_EXISTING and batch_log.exists():
    try:
        with open(batch_log) as f:
            for ln in f:
                try:
                    rec = json.loads(ln)
                    if rec.get('status') == 'ok':
                        already_done.add(rec.get('scene_idx', -1))
                except Exception:
                    pass
    except Exception:
        pass
    if already_done:
        print(f'  Resume: skipping {len(already_done)} already-completed scene(s)')

# --- Generate ─────────────────────────────────────────────────────────
results = []
batch_start = time.time()

with open(batch_log, 'a', buffering=1) as log_f:
    for i, scene in enumerate(valid_scenes):
        scene_idx = i + 1
        prompt = scene['prompt']
        img = scene.get('image')
        last_img = scene.get('last_image')
        canvas = scene['canvas']
        duration = scene['duration']
        steps = scene['steps']
        raw_seed = scene['seed']
        rewrite = scene.get('rewrite', False)

        # Seed: 0 or negative = random
        if raw_seed <= 0:
            seed = random.randint(1, 2**31 - 1)
        else:
            seed = int(raw_seed)

        # Skip if already done
        if scene_idx in already_done:
            print(f'  [{scene_idx:03d}/{len(valid_scenes)}] SKIP (already done)')
            results.append(('skipped', prompt, None))
            continue

        # Print scene info
        print(f'  [{scene_idx:03d}/{len(valid_scenes)}] seed={seed} {prompt[:60]}...')
        if img:
            print(f'         first_frame: {pathlib.Path(img).name}')
        if last_img:
            print(f'         last_frame:  {pathlib.Path(last_img).name}')
        print(f'         canvas: {canvas}, duration: {duration}s, steps: {steps}')

        t0 = time.time()
        try:
            out_path, report = generate_video(
                prompt=prompt,
                image_path=img,
                last_image_path=last_img,
                canvas=canvas,
                duration=duration,
                steps=steps,
                seed=seed,
                rewrite_prompt=rewrite,
                verbose=False,
            )
            elapsed = time.time() - t0
            sz = os.path.getsize(out_path) / 1024 / 1024
            print(f'    -> OK ({elapsed:.0f}s, {sz:.1f} MB, seed={report["seed"]})')
            results.append(('ok', prompt, out_path))

            # Copy to a named file for easy identification
            named_path = output_subdir / f'scene_{scene_idx:03d}.mp4'
            import shutil
            shutil.copy2(out_path, named_path)

            log_f.write(json.dumps({
                'scene_idx': scene_idx,
                'prompt': prompt,
                'status': 'ok',
                'elapsed_s': elapsed,
                'video': str(named_path),
                'original_video': out_path,
                'seed': report['seed'],
                'width': report['width'],
                'height': report['height'],
                'frames': report['num_frames'],
                'size_mb': sz,
                'canvas': canvas,
                'duration': duration,
                'steps': steps,
                'image': img,
                'last_image': last_img,
            }) + '\n')
        except Exception as e:
            elapsed = time.time() - t0
            print(f'    -> FAIL ({elapsed:.0f}s): {e}')
            traceback.print_exc(limit=2)
            results.append(('error', prompt, str(e)))
            log_f.write(json.dumps({
                'scene_idx': scene_idx,
                'prompt': prompt,
                'status': 'error',
                'error': str(e),
                'elapsed_s': elapsed,
                'seed': seed,
            }) + '\n')
        free_cuda()

total_elapsed = time.time() - batch_start
n_ok = sum(1 for r in results if r[0] == 'ok')
n_skip = sum(1 for r in results if r[0] == 'skipped')
n_err = sum(1 for r in results if r[0] == 'error')

print()
print('='*72)
print(f'Batch complete: {n_ok} ok / {n_skip} skipped / {n_err} errors in {total_elapsed:.0f}s')
print(f'  Output: {output_subdir}')
print(f'  Log:   {batch_log}')
print('='*72)

for f in sorted(output_subdir.glob('scene_*.mp4')):
    sz = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:>20s}  {sz:>8.1f} MB')

if n_ok > 0:
    print(f'\n  Tip: zip with `!cd {output_subdir} && zip -r batch.zip .`')
    print(f'  Tip: the JSONL log has all scene metadata for your records')
